In [1]:
import cv2
import os
import numpy as np
import pickle
from pathlib import Path
from time import time
import project_config as config

In [2]:
TRACKING_VIDEO_OUTPUT = './tracking_output/tracking_video.avi'
TRACKING_FRAME_FOLDER = './tracking_output/frames/'
Path(TRACKING_FRAME_FOLDER).mkdir(parents=True, exist_ok=True)

In [3]:
VIDEO_FILE = pickle.load(open(f'{config.DATA_PATH}/videopath.p', 'rb'))
SAVE_DETECTIONS = os.path.join(config.DATA_PATH, 'detections.p')
save_detections = pickle.load(open(SAVE_DETECTIONS, 'rb'))

In [4]:
cap = cv2.VideoCapture(VIDEO_FILE)
fps = cap.get(cv2.CAP_PROP_FPS)
video_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

In [5]:
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter(TRACKING_VIDEO_OUTPUT, fourcc, fps, (video_width, video_height))

In [6]:
player_trails = {}  
MAX_TRAIL_LENGTH = 20  
TRAIL_COLOR = (255, 255, 255)  
TRAIL_THICKNESS = 1

In [7]:
frame_count = 0
t = time()

In [ ]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    current_positions = {}  
    
    if frame_count + 1 in save_detections:
        boxes, scores, classes = save_detections[frame_count + 1]
        
        for box, score, cls_id in zip(boxes, scores, classes):
            # Get bottom-center point for trails
            bc_x = box.x
            bc_y = box.y
            
            # Determine if it's a player or ball
            is_ball = (cls_id == 37) or (hasattr(box, 'trackId') and box.trackId == 25)
            
            if not is_ball:
                player_id = getattr(box, 'trackId', cls_id)
                current_positions[player_id] = (bc_x, bc_y)
                
                if player_id not in player_trails:
                    player_trails[player_id] = []
                
                player_trails[player_id].append((bc_x, bc_y))
                
                if len(player_trails[player_id]) > MAX_TRAIL_LENGTH:
                    player_trails[player_id].pop(0)
            
            color = (0, 255, 0) if not is_ball else (0, 0, 255)  # Green for players, Red for ball
            cv2.rectangle(frame, 
                         (int(box.xLeft), int(box.yTop)), 
                         (int(box.xRight), int(box.yBottom)), 
                         color, 2)
            
            player_id = getattr(box, 'trackId', f"C{cls_id}")
            label = f"ID:{player_id}"
            if is_ball:
                label = "BALL"
            
            cv2.putText(frame, label, 
                       (int(box.xLeft), int(box.yTop) - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    
    for player_id, trail in player_trails.items():
        if len(trail) >= 2:
            for i in range(1, len(trail)):
                alpha = i / len(trail) 
                trail_color = tuple(int(c * alpha) for c in TRAIL_COLOR)
                
                cv2.line(frame, 
                        (int(trail[i-1][0]), int(trail[i-1][1])),
                        (int(trail[i][0]), int(trail[i][1])),
                        trail_color, TRAIL_THICKNESS)
    
    cv2.putText(frame, f"Frame: {frame_count}", (10, 30), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(frame, f"Players: {len(player_trails)}", (10, 60), 
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    cv2.imwrite(os.path.join(TRACKING_FRAME_FOLDER, f'track_{frame_count:06d}.jpg'), frame)
    out.write(frame)
    
    frame_count += 1
    if frame_count % 50 == 0:
        print(f"Processed {frame_count} frames...")

cap.release()
out.release()
print(f"Tracking video saved to: {TRACKING_VIDEO_OUTPUT}")
print(f"Total frames processed: {frame_count}")
print(f"Players tracked: {len(player_trails)}")

print("\nTracking Summary: ")
for player_id, trail in player_trails.items():
    print(f"Player {player_id}: {len(trail)} positions tracked")

Processed 50 frames...
Processed 100 frames...
Processed 150 frames...
Processed 200 frames...
Processed 250 frames...
Processed 300 frames...
Processed 350 frames...
Processed 400 frames...
Processed 450 frames...
Processed 500 frames...
Processed 550 frames...
Processed 600 frames...
Processed 650 frames...
Processed 700 frames...
Tracking video saved to: ./tracking_output/tracking_video.avi
Total frames processed: 721
Players tracked: 13

Tracking Summary: 
Player 8.0: 19 positions tracked
Player 12.0: 20 positions tracked
Player 13.0: 20 positions tracked
Player 14.0: 20 positions tracked
Player 16.0: 20 positions tracked
Player 17.0: 20 positions tracked
Player 15.0: 20 positions tracked
Player 18.0: 20 positions tracked
Player 21.0: 20 positions tracked
Player 19.0: 20 positions tracked
Player 20.0: 20 positions tracked
Player 28.0: 3 positions tracked
Player 32.0: 2 positions tracked
